The first step of our project was to analyze Pharmaly's website to understand its structure and determine the appropriate scraping strategy

import libraries

In [3]:
import requests #sends HTTP requests to retrieve website pages.
from bs4 import BeautifulSoup #parses HTML so we can inspect and extract elements
import pandas as pd

### Sending our first request

In [ ]:
url = "https://www.pharmaly-dz.com/"

response = requests.get(url, timeout=20) #timeout= 20 means that we allow up to 20 seconds for the request before timing out.
# stores the server's response.
print("Status:", response.status_code)
print("URL:", response.url)
print("HTML length:", len(response.text))

Status: 200
URL: https://www.pharmaly-dz.com/
HTML length: 557296


200 == request succesful we can access the website 

In [5]:
print(response.text[:2000])

<!doctype html>
<html lang="fr-FR">
<head><script data-optimized="1" id="wr-retarde-loader">(function(){var done=0,ev=["scroll","wheel","touchstart","pointerdown","mousemove","keydown","click"];function run(){var l=[].slice.call(document.querySelectorAll('script[type="wr/retarde"]'));(function next(i){if(i>=l.length){document.dispatchEvent(new Event("wr:retardes"));return}var o=l[i],s=document.createElement("script");for(var k=0;k<o.attributes.length;k++){var a=o.attributes[k];if(a.name!=="type"&&a.name!=="data-wr-src")s.setAttribute(a.name,a.value);}if(o.getAttribute("data-wr-src")){s.src=o.getAttribute("data-wr-src");s.async=!1;s.onload=s.onerror=function(){next(i+1)};o.parentNode.replaceChild(s,o)}else{s.text=o.text;o.parentNode.replaceChild(s,o);next(i+1)}})(0)}function go(){if(done)return;done=1;ev.forEach(function(e){removeEventListener(e,go,{passive:!0})});if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",run)}else{run()}}ev.forEach(function(e){add

### Parsing the HTML

"lxml" :A parser reads HTML and organizes it into a tree-like structure that Python can navigate.
HTML can contain nested elements: The parser understands the relationships between these elements.

In [ ]:
soup = BeautifulSoup(response.text, "lxml")

print(soup.title)
print(soup.title.text) # only the textual content of the title extracted

<title>Votre Parapharmacie en ligne en Algérie | Pharmaly</title>
Votre Parapharmacie en ligne en Algérie | Pharmaly


### Getting an overview of the homepage: 

In [ ]:
print("Page title:", soup.title.text)
print("Number of links:", len(soup.find_all("a"))) # finds every HTML anchor element
print("Number of images:", len(soup.find_all("img"))) 
print("Number of divs:", len(soup.find_all("div"))) # generic HTML container. Websites use divs to organize and group elements into sections and other components.

Page title: Votre Parapharmacie en ligne en Algérie | Pharmaly
Number of links: 165
Number of images: 57
Number of divs: 1107


165 links: The homepage contains 165 HTML anchor elements. These may link to categories, products, pages, or external resources.



57 images: There are 57 image elements. Some may be product images, banners, logos, or icons.



1,107 divs: There are 1,107 container elements. These help organize the page's content.

In [8]:
links = soup.find_all("a")

for link in links[:30]:
    print("Text:", link.get_text(strip=True))
    print("URL:", link.get("href"))
    print("-" * 50)

Text: Aller au contenu
URL: #content
--------------------------------------------------
Text: 
URL: https://www.pharmaly-dz.com
--------------------------------------------------
Text: 
URL: https://www.pharmaly-dz.com/
--------------------------------------------------
Text: 
URL: tel:+213560417300
--------------------------------------------------
Text: 0560 41 73 00
URL: tel:+213560417300
--------------------------------------------------
Text: 
URL: /mon-compte/orders/
--------------------------------------------------
Text: Mon compte
URL: /mon-compte/orders/
--------------------------------------------------
Text: DA0,000Panier
URL: #
--------------------------------------------------
Text: Santé
URL: https://www.pharmaly-dz.com/categorie/sante/
--------------------------------------------------
Text: Visage
URL: https://www.pharmaly-dz.com/categorie/visage/
--------------------------------------------------
Text: Corps
URL: https://www.pharmaly-dz.com/categorie/corps/
----------

In [ ]:
from urllib.parse import urljoin
# Its purpose is to combine a base URL with a relative URL urljoin(base_url, relative_url)
category_links = {}

for link in soup.find_all("a", href=True):
    href = link.get("href") # retrieves the value of the HTML attribute named href specifies the destination URL of a link 
   
    if "/categorie/" in href:
        category_url = urljoin(response.url, href)
        category_name = link.get_text(strip=True)

        category_links[category_url] = category_name

for url, name in category_links.items():
    print(f"{name}: {url}")

print("Total unique category URLs:", len(category_links))

Santé: https://www.pharmaly-dz.com/categorie/sante/
: https://www.pharmaly-dz.com/categorie/visage/
Corps: https://www.pharmaly-dz.com/categorie/corps/
: https://www.pharmaly-dz.com/categorie/cheveux/
K-Beauty: https://www.pharmaly-dz.com/categorie/k-beauty/
: https://www.pharmaly-dz.com/categorie/solaire/
: https://www.pharmaly-dz.com/categorie/hygiene/
: https://www.pharmaly-dz.com/categorie/maquillage-et-parfums/
Total unique category URLs: 8


In [10]:
for link in soup.find_all("a", href=True):
    if "/categorie/" in link.get("href") and not link.get_text(strip=True):
        print(link.prettify()[:1000])
        print("-" * 60)

<a class="jet-engine-listing-overlay-link" href="https://www.pharmaly-dz.com/categorie/visage/">
</a>

------------------------------------------------------------
<a class="jet-engine-listing-overlay-link" href="https://www.pharmaly-dz.com/categorie/cheveux/">
</a>

------------------------------------------------------------
<a class="jet-engine-listing-overlay-link" href="https://www.pharmaly-dz.com/categorie/solaire/">
</a>

------------------------------------------------------------
<a class="jet-engine-listing-overlay-link" href="https://www.pharmaly-dz.com/categorie/hygiene/">
</a>

------------------------------------------------------------
<a class="jet-engine-listing-overlay-link" href="https://www.pharmaly-dz.com/categorie/maquillage-et-parfums/">
</a>

------------------------------------------------------------


In [11]:
for link in soup.find_all("a", class_="jet-engine-listing-overlay-link"):
    print(link.parent.prettify()[:2000])
    break

<div class="jet-engine-listing-overlay-wrap" data-url="https://www.pharmaly-dz.com/">
 <div class="elementor elementor-14763" data-elementor-id="14763" data-elementor-post-type="jet-engine" data-elementor-type="jet-listing-items">
  <div class="elementor-element elementor-element-c4bfb74 e-flex e-con-boxed e-con e-parent" data-e-type="container" data-element_type="container" data-id="c4bfb74">
   <div class="e-con-inner">
    <div class="elementor-element elementor-element-662ba44 elementor-widget__width-initial elementor-widget elementor-widget-image" data-e-type="widget" data-element_type="widget" data-id="662ba44" data-widget_type="image.default">
     <div class="elementor-widget-container">
      <picture class="attachment-thumbnail size-thumbnail wp-image-13633">
       <source sizes="(max-width: 150px) 100vw, 150px" srcset="https://www.pharmaly-dz.com/wp-content/uploads/2025/02/pharmaly-dz-dektop.jpeg.001.jpeg.webp 1219w, https://www.pharmaly-dz.com/wp-content/uploads/2025/02/ph

In [12]:
for link in soup.find_all("a", class_="jet-engine-listing-overlay-link"):
    card = link.parent

    print("Category URL:", link.get("href"))
    print("Card text:", card.get_text(" ", strip=True))
    print("-" * 60)

    break

Category URL: https://www.pharmaly-dz.com/
Card text: Vente de produits para pharmaceutiques en Algérie
------------------------------------------------------------


In [13]:
for link in soup.find_all("a", class_="jet-engine-listing-overlay-link"):
    href = link.get("href", "")

    if "/categorie/" in href:
        card = link.parent

        print("Category URL:", href)
        print("Card text:", card.get_text(" ", strip=True))
        print("-" * 60)

Category URL: https://www.pharmaly-dz.com/categorie/visage/
Card text: Visage
------------------------------------------------------------
Category URL: https://www.pharmaly-dz.com/categorie/cheveux/
Card text: Cheveux
------------------------------------------------------------
Category URL: https://www.pharmaly-dz.com/categorie/solaire/
Card text: Solaire
------------------------------------------------------------
Category URL: https://www.pharmaly-dz.com/categorie/hygiene/
Card text: Hygiène
------------------------------------------------------------
Category URL: https://www.pharmaly-dz.com/categorie/maquillage-et-parfums/
Card text: Maquillage
------------------------------------------------------------


Our investigation revealed an important detail about the website's HTML:
- Some category links contain their names directly inside the <a> element.
- Others use an overlay link with no text, while the category name is stored in its parent container.
- We can retrieve those names using link.parent.get_text()

# Combine both extraction methods

In [14]:
category_links = {}

for link in soup.find_all("a", href=True):
    href = link.get("href", "")

    if "/categorie/" in href:
        category_url = urljoin(response.url, href)

        # First, try to extract the text directly from the link
        category_name = link.get_text(strip=True)

        # If the link has no text, extract it from its parent
        if not category_name:
            category_name = link.parent.get_text(" ", strip=True)

        category_links[category_url] = category_name

for url, name in category_links.items():
    print(f"{name}: {url}")

print("\nTotal unique category URLs:", len(category_links))

Santé: https://www.pharmaly-dz.com/categorie/sante/
Visage: https://www.pharmaly-dz.com/categorie/visage/
Corps: https://www.pharmaly-dz.com/categorie/corps/
Cheveux: https://www.pharmaly-dz.com/categorie/cheveux/
K-Beauty: https://www.pharmaly-dz.com/categorie/k-beauty/
Solaire: https://www.pharmaly-dz.com/categorie/solaire/
Hygiène: https://www.pharmaly-dz.com/categorie/hygiene/
Maquillage: https://www.pharmaly-dz.com/categorie/maquillage-et-parfums/

Total unique category URLs: 8


## 2. Website Categories

The homepage contains 8 unique product category URLs.

The categories were extracted using BeautifulSoup by searching for anchor elements containing `/categorie/` in their href attribute.

Two HTML structures were identified:
- Category names directly inside the anchor element.
- Category names inside the parent element of an overlay link.

The extraction handles both cases and uses `urljoin()` to convert relative URLs into absolute URLs.

In [16]:
category_url = "https://www.pharmaly-dz.com/categorie/sante/"

category_response = requests.get(category_url, timeout=20)

print("Status:", category_response.status_code)
print("URL:", category_response.url)
print("HTML length:", len(category_response.text))

Status: 200
URL: https://www.pharmaly-dz.com/categorie/sante/
HTML length: 1153936


In [17]:
category_soup = BeautifulSoup(category_response.text, "lxml")

print("Page title:", category_soup.title.get_text(strip=True))
print("Number of product links:", len(
    category_soup.find_all("a", href=lambda href: href and "/produit/" in href)
))

Page title: Santé | Pharmaly
Number of product links: 38


In [19]:
product_links = category_soup.find_all(
    "a",
    href=lambda href: href and "/produit/" in href
)

print("Number of product links:", len(product_links))

Number of product links: 38


In [20]:
print(product_links[0].prettify()[:2000])

<a class="elementor-button elementor-button-link elementor-size-sm" href="/produit/carte-cadeau/">
 <span class="elementor-button-content-wrapper">
  <span class="elementor-button-icon">
   <svg height="300" id="svg5073" viewbox="0 0 682.66669 682.66669" width="300" xml:space="preserve" xmlns="http://www.w3.org/2000/svg" xmlns:svgjs="http://svgjs.dev/svgjs" xmlns:xlink="http://www.w3.org/1999/xlink">
    <g height="100%" transform="matrix(1,0,0,1,0,0)" width="100%">
     <defs id="defs5077">
      <clippath clippathunits="userSpaceOnUse" id="clipPath5087">
       <path d="M 0,512 H 512 V 0 H 0 Z" data-original-color="#000000ff" fill="#ffffff" fill-opacity="1" id="path5085" stroke="none" stroke-opacity="1">
       </path>
      </clippath>
     </defs>
     <g id="g5079" transform="matrix(1.3333333,0,0,-1.3333333,0,682.66667)">
      <g id="g5081">
       <g clip-path="url(#clipPath5087)" id="g5083">
        <g id="g5089" transform="translate(472,376)">
         <path d="m 0,0 h -382 c 

In [21]:
for i, link in enumerate(product_links[:20]):
    print("Index:", i)
    print("URL:", link.get("href"))
    print("Class:", link.get("class"))
    print("Text:", link.get_text(" ", strip=True))
    print("-" * 50)

Index: 0
URL: /produit/carte-cadeau/
Class: ['elementor-button', 'elementor-button-link', 'elementor-size-sm']
Text: Cadeaux
--------------------------------------------------
Index: 1
URL: https://www.pharmaly-dz.com/produit/effluvium-croissance-vitalite-du-cheveu-168-gel/
Class: ['jet-listing-dynamic-image__link']
Text: 
--------------------------------------------------
Index: 2
URL: https://www.pharmaly-dz.com/produit/effluvium-croissance-vitalite-du-cheveu-168-gel/
Class: None
Text: Laboratoires NHCO Nutrition
--------------------------------------------------
Index: 3
URL: https://www.pharmaly-dz.com/produit/effluvium-croissance-vitalite-du-cheveu-168-gel/
Class: None
Text: Effluvium Croissance Vitalité Du Cheveu 168 Gel
--------------------------------------------------
Index: 4
URL: https://www.pharmaly-dz.com/produit/forcapil-anti-chute-90-comprmes/
Class: ['jet-listing-dynamic-image__link']
Text: 
--------------------------------------------------
Index: 5
URL: https://www.ph

Conclusion: Counting product links directly overestimates the number of products because each product generates multiple links.
 Our next goal is to identify the common parent container that groups the image, brand, name, and price into one product card.


### Find the product card container


In [23]:
product_link = product_links[3]

for level, parent in enumerate(product_link.parents):
    print("Level:", level)
    print("Tag:", parent.name)
    print("Classes:", parent.get("class"))
    print("Text:", parent.get_text(" ", strip=True)[:300])
    print("-" * 50)

    if level == 6:
        break

Level: 0
Tag: p
Classes: ['elementor-heading-title', 'elementor-size-default']
Text: Effluvium Croissance Vitalité Du Cheveu 168 Gel
--------------------------------------------------
Level: 1
Tag: div
Classes: ['elementor-widget-container']
Text: Effluvium Croissance Vitalité Du Cheveu 168 Gel
--------------------------------------------------
Level: 2
Tag: div
Classes: ['elementor-element', 'elementor-element-58faab4', 'elementor-widget__width-inherit', 'elementor-widget', 'elementor-widget-heading']
Text: Effluvium Croissance Vitalité Du Cheveu 168 Gel
--------------------------------------------------
Level: 3
Tag: div
Classes: ['e-con-inner']
Text: Laboratoires NHCO Nutrition Effluvium Croissance Vitalité Du Cheveu 168 Gel Promo -1600 DA DA 11.500,00 Le prix initial était : DA 11.500,00. DA 9.900,00 Le prix actuel est : DA 9.900,00. J'ACHÈTE
--------------------------------------------------
Level: 4
Tag: div
Classes: ['elementor-element', 'elementor-element-94e2b60', 'e-flex', 'e

In [24]:
product_card = product_link.find_parent(
    "div",
    class_="jet-listing-grid__item"
)

print(product_card.prettify()[:12000])

<div class="jet-listing-grid__item jet-listing-dynamic-post-10413 jet-equal-columns elementor-dcss-1825310323749550" data-post-id="10413">
 <div class="elementor elementor-362" data-elementor-id="362" data-elementor-post-type="jet-engine" data-elementor-type="jet-listing-items">
  <div class="elementor-element elementor-element-94e2b60 e-flex e-con-boxed e-con e-parent" data-e-type="container" data-element_type="container" data-id="94e2b60" data-settings='{"background_background":"classic"}'>
   <div class="e-con-inner">
    <div class="elementor-element elementor-element-7901bad wr_square_image elementor-widget elementor-widget-jet-listing-dynamic-image" data-e-type="widget" data-element_type="widget" data-id="7901bad" data-widget_type="jet-listing-dynamic-image.default">
     <div class="elementor-widget-container">
      <div class="jet-listing jet-listing-dynamic-image">
       <a class="jet-listing-dynamic-image__link" href="https://www.pharmaly-dz.com/produit/effluvium-croissance

In [25]:
# Select the product card
product_card = product_link.find_parent(
    "div",
    class_="jet-listing-grid__item"
)

# Extract product information
product_url = product_card.select_one(
    "a.jet-listing-dynamic-image__link"
)["href"]

image_url = product_card.select_one(
    "img.jet-listing-dynamic-image__img"
)["src"]

product_name = product_card.select_one(
    "p.elementor-heading-title a"
).get_text(strip=True)

prices = product_card.select_one("p.price")

original_price = prices.select_one("del").get_text(" ", strip=True)
current_price = prices.select_one("ins").get_text(" ", strip=True)

print("Product URL:", product_url)
print("Image URL:", image_url)
print("Product name:", product_name)
print("Original price:", original_price)
print("Current price:", current_price)

Product URL: https://www.pharmaly-dz.com/produit/effluvium-croissance-vitalite-du-cheveu-168-gel/
Image URL: https://www.pharmaly-dz.com/wp-content/uploads/2024/11/effluvium-croissance-vitalite-du-cheveu-168-gel-600x600.webp
Product name: Effluvium Croissance Vitalité Du Cheveu 168 Gel
Original price: DA 11.500,00
Current price: DA 9.900,00


In [26]:
# Extract the brand
brand_element = product_card.select_one(
    "span.elementor-heading-title a"
)

brand = brand_element.get_text(strip=True) if brand_element else None

print("Brand:", brand)

Brand: Laboratoires NHCO Nutrition


In [27]:
# Select another product's name link
second_product_link = product_links[6]

# Find its product card
second_product_card = second_product_link.find_parent(
    "div",
    class_="jet-listing-grid__item"
)

# Extract product name
name_element = second_product_card.select_one(
    "p.elementor-heading-title a"
)

print("Product name:", name_element.get_text(strip=True))

# Extract brand
brand_element = second_product_card.select_one(
    "span.elementor-heading-title a"
)

print("Brand:", brand_element.get_text(strip=True) if brand_element else None)

# Extract prices
prices = second_product_card.select_one("p.price")

print("Price element found:", prices is not None)
print("Price HTML:")
print(prices.prettify() if prices else "No price found")


Product name: Forcapil Anti Chute 90 Comprimés
Brand: Forcapil
Price element found: True
Price HTML:
<p class="price">
 <del aria-hidden="true">
  <span class="woocommerce-Price-amount amount">
   <bdi>
    <span class="woocommerce-Price-currencySymbol" translate="no">
     DA
    </span>
    9.000,00
   </bdi>
  </span>
 </del>
 <span class="screen-reader-text">
  Le prix initial était : DA 9.000,00.
 </span>
 <ins aria-hidden="true">
  <span class="woocommerce-Price-amount amount">
   <bdi>
    <span class="woocommerce-Price-currencySymbol" translate="no">
     DA
    </span>
    8.200,00
   </bdi>
  </span>
 </ins>
 <span class="screen-reader-text">
  Le prix actuel est : DA 8.200,00.
 </span>
</p>



In [30]:
for link in product_links:
    card = link.find_parent("div", class_="jet-listing-grid__item")

    if card is None:
        continue

    name_element = card.select_one("p.elementor-heading-title a")

    if name_element is None:
        continue

    name = name_element.get_text(strip=True)

    price = card.select_one("p.price")

    original = price.select_one("del") if price else None
    current = price.select_one("ins") if price else None

    print("Product:", name)
    print("Has original price:", original is not None)
    print("Has current price:", current is not None)
    print("-" * 50)

Product: Effluvium Croissance Vitalité Du Cheveu 168 Gel
Has original price: True
Has current price: True
--------------------------------------------------
Product: Effluvium Croissance Vitalité Du Cheveu 168 Gel
Has original price: True
Has current price: True
--------------------------------------------------
Product: Effluvium Croissance Vitalité Du Cheveu 168 Gel
Has original price: True
Has current price: True
--------------------------------------------------
Product: Forcapil Anti Chute 90 Comprimés
Has original price: True
Has current price: True
--------------------------------------------------
Product: Forcapil Anti Chute 90 Comprimés
Has original price: True
Has current price: True
--------------------------------------------------
Product: Forcapil Anti Chute 90 Comprimés
Has original price: True
Has current price: True
--------------------------------------------------
Product: Forcapil Anti Chute 90 Comprimés
Has original price: True
Has current price: True
------------

In [29]:
# Store unique product cards
unique_cards = {}

for link in product_links:
    card = link.find_parent(
        "div",
        class_="jet-listing-grid__item"
    )

    if card is not None:
        product_id = card.get("data-post-id")

        if product_id:
            unique_cards[product_id] = card

print("Total product links:", len(product_links))
print("Unique product cards:", len(unique_cards))

Total product links: 38
Unique product cards: 12


In [31]:
for product_id, card in list(unique_cards.items())[:10]:

    name_element = card.select_one("p.elementor-heading-title a")
    name = name_element.get_text(strip=True) if name_element else None

    price = card.select_one("p.price")

    print("Product ID:", product_id)
    print("Product:", name)
    print("Price HTML:", price.get_text(" ", strip=True) if price else "No price element")
    print("-" * 50)
    

Product ID: 10413
Product: Effluvium Croissance Vitalité Du Cheveu 168 Gel
Price HTML: DA 11.500,00 Le prix initial était : DA 11.500,00. DA 9.900,00 Le prix actuel est : DA 9.900,00.
--------------------------------------------------
Product ID: 11792
Product: Forcapil Anti Chute 90 Comprimés
Price HTML: DA 9.000,00 Le prix initial était : DA 9.000,00. DA 8.200,00 Le prix actuel est : DA 8.200,00.
--------------------------------------------------
Product ID: 70502
Product: Biogaia Gouttesprotectis   10ml
Price HTML: DA 5.750,00
--------------------------------------------------
Product ID: 69900
Product: Fortimel Saveur Peche Mangue 2.4 Kcal   200ml
Price HTML: DA 740,00
--------------------------------------------------
Product ID: 60889
Product: Acm Novophane Complément Alimentaire Ongles/cheveux B/180   108g
Price HTML: DA 6.500,00
--------------------------------------------------
Product ID: 54904
Product: Cheks Postbiotic Balance  Bt/45
Price HTML: DA 2.700,00
-----------------

### Price Structure

The website has a dedicated Promotions section.

Two pricing structures were identified:

1. Promotional products:
   - Original price is stored inside the `<del>` element.
   - Current discounted price is stored inside the `<ins>` element.

2. Regular products:
   - Only one price is displayed directly inside the `<p class="price">` element.
   - No original price is provided.

Therefore, the original price is an optional field, while the current price is the price displayed for every product examined.

In [32]:
biogaia_card = unique_cards["70502"]

price_element = biogaia_card.select_one("p.price")

print(price_element.prettify())

<p class="price">
 <span class="woocommerce-Price-amount amount">
  <bdi>
   <span class="woocommerce-Price-currencySymbol" translate="no">
    DA
   </span>
   5.750,00
  </bdi>
 </span>
</p>



In [33]:
for product_id, card in list(unique_cards.items())[:10]:

    name = card.select_one(
        "p.elementor-heading-title a"
    ).get_text(strip=True)

    price = card.select_one("p.price")

    original_element = price.select_one("del") if price else None
    current_element = price.select_one("ins") if price else None

    if original_element and current_element:
        original_price = original_element.get_text(" ", strip=True)
        current_price = current_element.get_text(" ", strip=True)

    elif price:
        original_price = None
        current_price = price.get_text(" ", strip=True)

    else:
        original_price = None
        current_price = None

    print("Product:", name)
    print("Original price:", original_price)
    print("Current price:", current_price)
    print("-" * 50)
    

Product: Effluvium Croissance Vitalité Du Cheveu 168 Gel
Original price: DA 11.500,00
Current price: DA 9.900,00
--------------------------------------------------
Product: Forcapil Anti Chute 90 Comprimés
Original price: DA 9.000,00
Current price: DA 8.200,00
--------------------------------------------------
Product: Biogaia Gouttesprotectis   10ml
Original price: None
Current price: DA 5.750,00
--------------------------------------------------
Product: Fortimel Saveur Peche Mangue 2.4 Kcal   200ml
Original price: None
Current price: DA 740,00
--------------------------------------------------
Product: Acm Novophane Complément Alimentaire Ongles/cheveux B/180   108g
Original price: None
Current price: DA 6.500,00
--------------------------------------------------
Product: Cheks Postbiotic Balance  Bt/45
Original price: None
Current price: DA 2.700,00
--------------------------------------------------
Product: Granions Biotine Vitamine B8 46g
Original price: None
Current price: DA 4.

**Validation:**

The extraction logic was tested on 10 unique products from the Santé category:
- 2 promotional products with original and current prices.
- 8 regular products with only a current price.

Both pricing structures were extracted successfully.

In [34]:
for product_id, card in list(unique_cards.items())[:10]:

    name_element = card.select_one("p.elementor-heading-title a")
    brand_element = card.select_one("span.elementor-heading-title a")
    image_element = card.select_one("img.jet-listing-dynamic-image__img")

    print("Product:", name_element.get_text(strip=True) if name_element else None)
    print("Brand:", brand_element.get_text(strip=True) if brand_element else None)
    print("Image URL:", image_element.get("src") if image_element else None)
    print("-" * 60)

Product: Effluvium Croissance Vitalité Du Cheveu 168 Gel
Brand: Laboratoires NHCO Nutrition
Image URL: https://www.pharmaly-dz.com/wp-content/uploads/2024/11/effluvium-croissance-vitalite-du-cheveu-168-gel-600x600.webp
------------------------------------------------------------
Product: Forcapil Anti Chute 90 Comprimés
Brand: Forcapil
Image URL: https://www.pharmaly-dz.com/wp-content/uploads/2024/11/64072-e-forcapilantichute90cp-offrepromo-fr-face-600x600.webp
------------------------------------------------------------
Product: Biogaia Gouttesprotectis   10ml
Brand: Biogaia
Image URL: None
------------------------------------------------------------
Product: Fortimel Saveur Peche Mangue 2.4 Kcal   200ml
Brand: Fortimel
Image URL: None
------------------------------------------------------------
Product: Acm Novophane Complément Alimentaire Ongles/cheveux B/180   108g
Brand: Laboratoires ACM
Image URL: None
------------------------------------------------------------
Product: Cheks Po

In [40]:
# Search for pagination elements
pagination_elements = category_soup.find_all(
    ["nav", "div", "ul"],
    class_=lambda classes: classes and any(
        "pagination" in class_name.lower()
        for class_name in classes
    )
)

print("Pagination elements found:", len(pagination_elements))

for element in pagination_elements:
    print(element.prettify()[:3000])
    print("-" * 60)

Pagination elements found: 0


In [41]:
# Look for links that might lead to another category page
for link in category_soup.find_all("a", href=True):

    href = link["href"]

    if (
        "page/" in href
        or "paged=" in href
        or "next" in str(link.get("class", [])).lower()
    ):
        print("Text:", link.get_text(" ", strip=True))
        print("URL:", href)
        print("Class:", link.get("class"))
        print("-" * 50)

In [42]:
# Inspect the product listing container
first_card = next(iter(unique_cards.values()))

listing_container = first_card.parent

print("Container tag:", listing_container.name)
print("Container classes:", listing_container.get("class"))
print("Number of direct product cards:", len(
    listing_container.find_all(
        "div",
        class_="jet-listing-grid__item",
        recursive=False
    )
))
print(listing_container.prettify()[:3000])

Container tag: div
Container classes: ['jet-listing-grid__items', 'grid-col-desk-5', 'grid-col-tablet-3', 'grid-col-mobile-2', 'jet-listing-grid--362', 'jet-listing-grid__scroll-slider-wrap-desktop', 'jet-listing-grid__scroll-slider-wrap-tablet', 'jet-listing-grid__scroll-slider-wrap-mobile', 'jet-equal-columns__wrapper', 'woocommerce']
Number of direct product cards: 2
<div class="jet-listing-grid__items grid-col-desk-5 grid-col-tablet-3 grid-col-mobile-2 jet-listing-grid--362 jet-listing-grid__scroll-slider-wrap-desktop jet-listing-grid__scroll-slider-wrap-tablet jet-listing-grid__scroll-slider-wrap-mobile jet-equal-columns__wrapper woocommerce" data-listing-id="362" data-listing-source="query" data-nav='{"enabled":false,"type":null,"more_el":null,"query":{"query_id":10,"custom_query_id":10,"posts_per_page":10,"allowed_query_ids":[10],"signature":"78c4a69b543593ab36b365fa2bb8ff5db489cb8646c68828a48312a0b901e5c0"},"widget_settings":{"lisitng_id":362,"posts_num":10,"columns":5,"columns

In [43]:
# Count all product cards inside this listing container
cards_inside_container = listing_container.find_all(
    "div",
    class_="jet-listing-grid__item"
)

print("Product cards inside this container:", len(cards_inside_container))

print("\nDirect children:")
for child in listing_container.find_all("div", recursive=False):
    print("Tag:", child.name)
    print("Classes:", child.get("class"))
    print("-" * 40)

Product cards inside this container: 2

Direct children:
Tag: div
Classes: ['jet-listing-grid__item', 'jet-listing-dynamic-post-10413', 'jet-equal-columns', 'elementor-dcss-1825310323749550']
----------------------------------------
Tag: div
Classes: ['jet-listing-grid__item', 'jet-listing-dynamic-post-11792', 'jet-equal-columns', 'elementor-dcss-1825310373230224']
----------------------------------------


In [44]:
# Find all product listing containers on the category page
listing_containers = category_soup.find_all(
    "div",
    class_="jet-listing-grid__items"
)

print("Number of listing containers:", len(listing_containers))

for i, container in enumerate(listing_containers, start=1):

    cards = container.find_all(
        "div",
        class_="jet-listing-grid__item",
        recursive=False
    )

    print(f"\nContainer {i}")
    print("Classes:", container.get("class"))
    print("Number of products:", len(cards))

    for card in cards:
        name_element = card.select_one("p.elementor-heading-title a")
        name = name_element.get_text(strip=True) if name_element else "Name not found"
        print("-", name)
        

Number of listing containers: 3

Container 1
Classes: ['jet-listing-grid__items', 'grid-col-desk-5', 'grid-col-tablet-4', 'grid-col-mobile-2', 'jet-listing-grid--764', 'jet-listing-grid__scroll-slider-wrap-desktop', 'jet-listing-grid__scroll-slider-wrap-tablet', 'jet-listing-grid__scroll-slider-wrap-mobile', 'jet-equal-columns__wrapper']
Number of products: 2
- Name not found
- Name not found

Container 2
Classes: ['jet-listing-grid__items', 'grid-col-desk-5', 'grid-col-tablet-3', 'grid-col-mobile-2', 'jet-listing-grid--362', 'jet-listing-grid__scroll-slider-wrap-desktop', 'jet-listing-grid__scroll-slider-wrap-tablet', 'jet-listing-grid__scroll-slider-wrap-mobile', 'jet-equal-columns__wrapper', 'woocommerce']
Number of products: 2
- Effluvium Croissance Vitalité Du Cheveu 168 Gel
- Forcapil Anti Chute 90 Comprimés

Container 3
Classes: ['jet-listing-grid__items', 'grid-col-desk-5', 'grid-col-tablet-3', 'grid-col-mobile-2', 'jet-listing-grid--362', 'jet-equal-columns__wrapper', 'woocomm

In [45]:
for i, container in enumerate(listing_containers, start=1):

    print(f"\nContainer {i}")
    print("data-page:", container.get("data-page"))
    print("data-pages:", container.get("data-pages"))
    print("data-listing-id:", container.get("data-listing-id"))

    nav = container.get("data-nav")

    if nav:
        import json
        nav_data = json.loads(nav)

        print("Navigation enabled:", nav_data.get("enabled"))
        print("Posts per page:", nav_data.get("query", {}).get("posts_per_page"))
        print("Load more:", nav_data.get("widget_settings", {}).get("use_load_more"))

    print("-" * 50)


Container 1
data-page: 1
data-pages: 1
data-listing-id: 764
Navigation enabled: False
Posts per page: 0
Load more: 
--------------------------------------------------

Container 2
data-page: 1
data-pages: 1
data-listing-id: 362
Navigation enabled: False
Posts per page: 10
Load more: 
--------------------------------------------------

Container 3
data-page: 1
data-pages: 17
data-listing-id: 362
Navigation enabled: True
Posts per page: 10
Load more: yes
--------------------------------------------------


In [46]:
import json

container = listing_containers[2]

nav_data = json.loads(container.get("data-nav"))

print(json.dumps(nav_data, indent=4, ensure_ascii=False))

{
    "enabled": true,
    "type": "scroll",
    "more_el": null,
    "query": {
        "query_id": 6,
        "custom_query_id": 6,
        "posts_per_page": 10,
        "allowed_query_ids": [
            6
        ],
        "signature": "e0b6aac34f8e2e3a23d77209e4cad0479efef752e797a350fd7f0e3890b62b0e"
    },
    "widget_settings": {
        "lisitng_id": 362,
        "posts_num": 10,
        "columns": 5,
        "columns_tablet": 3,
        "columns_mobile": 2,
        "column_min_width": 240,
        "column_min_width_tablet": 240,
        "column_min_width_mobile": 240,
        "inline_columns_css": false,
        "is_archive_template": "yes",
        "post_status": [
            "publish"
        ],
        "use_random_posts_num": "",
        "max_posts_num": 9,
        "not_found_message": "Aucun produit trouvé",
        "is_masonry": false,
        "equal_columns_height": "yes",
        "use_load_more": "yes",
        "load_more_id": "",
        "load_more_type": "scroll",
 

 the Santé page uses JetEngine's infinite-scroll mechanism for its main product listing. A normal requests.get() retrieves the initial HTML, but it does not automatically execute the JavaScript responsible for loading subsequent batches.

In [47]:
# Find JavaScript files related to JetEngine or listing functionality
scripts = category_soup.find_all("script", src=True)

for script in scripts:
    src = script.get("src", "")

    if "jet" in src.lower() or "elementor" in src.lower():
        print(src)
        

In [48]:
# Count external and inline JavaScript scripts
external_scripts = category_soup.find_all("script", src=True)
inline_scripts = category_soup.find_all(
    "script",
    src=False
)

print("External scripts:", len(external_scripts))
print("Inline scripts:", len(inline_scripts))

# Search inline scripts for JetEngine or load-more configuration
for i, script in enumerate(inline_scripts):

    content = script.get_text()

    if "jet" in content.lower() or "load_more" in content.lower():
        print(f"\nInline script {i}")
        print(content[:1500])
        print("-" * 50)

External scripts: 39
Inline scripts: 57

Inline script 0
(function(){var done=0,ev=["scroll","wheel","touchstart","pointerdown","mousemove","keydown","click"];function run(){var l=[].slice.call(document.querySelectorAll('script[type="wr/retarde"]'));(function next(i){if(i>=l.length){document.dispatchEvent(new Event("wr:retardes"));return}var o=l[i],s=document.createElement("script");for(var k=0;k<o.attributes.length;k++){var a=o.attributes[k];if(a.name!=="type"&&a.name!=="data-wr-src")s.setAttribute(a.name,a.value);}if(o.getAttribute("data-wr-src")){s.src=o.getAttribute("data-wr-src");s.async=!1;s.onload=s.onerror=function(){next(i+1)};o.parentNode.replaceChild(s,o)}else{s.text=o.text;o.parentNode.replaceChild(s,o);next(i+1)}})(0)}function go(){if(done)return;done=1;ev.forEach(function(e){removeEventListener(e,go,{passive:!0})});if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",run)}else{run()}}ev.forEach(function(e){addEventListener(e,go,{passive:!0})});

In [49]:
# Find scripts whose loading is delayed
delayed_scripts = category_soup.find_all(
    "script",
    attrs={"data-wr-src": True}
)

print("Delayed scripts found:", len(delayed_scripts))

for script in delayed_scripts:
    print("Type:", script.get("type"))
    print("URL:", script.get("data-wr-src"))
    print("-" * 60)

Delayed scripts found: 9
Type: wr/retarde
URL: https://www.googletagmanager.com/gtag/js?id=G-P0DY65X7RF
------------------------------------------------------------
Type: wr/retarde
URL: https://www.pharmaly-dz.com/wp-content/plugins/jet-popup/assets/js/lib/anime-js/anime.min.js?ver=2.0.2
------------------------------------------------------------
Type: wr/retarde
URL: https://www.pharmaly-dz.com/wp-content/plugins/jet-popup/assets/js/jet-popup-frontend.js?ver=2.2.1
------------------------------------------------------------
Type: wr/retarde
URL: https://www.pharmaly-dz.com/wp-content/plugins/pixelyoursite/dist/scripts/jquery.bind-first-0.2.3.min.js?ver=0.2.3
------------------------------------------------------------
Type: wr/retarde
URL: https://www.pharmaly-dz.com/wp-content/plugins/pixelyoursite/dist/scripts/js.cookie-2.1.3.min.js?ver=2.1.3
------------------------------------------------------------
Type: wr/retarde
URL: https://www.pharmaly-dz.com/wp-content/plugins/pixelyours

In [50]:
# Search all inline scripts for AJAX and listing-related keywords
keywords = [
    "jet_engine_ajax",
    "jet_engine_load",
    "load_more",
    "infinite_scroll",
    "admin-ajax.php",
    "query_id"
]

for i, script in enumerate(inline_scripts):
    content = script.get_text()

    matches = [word for word in keywords if word in content.lower()]

    if matches:
        print(f"\nScript {i}")
        print("Keywords found:", matches)

        for word in matches:
            position = content.lower().find(word)
            print(content[max(0, position - 200):position + 500])
            print("-" * 40)


Script 24
Keywords found: ['admin-ajax.php']
var wc_add_to_cart_params={"ajax_url":"/wp-admin/admin-ajax.php","wc_ajax_url":"/?wc-ajax=%%endpoint%%","i18n_view_cart":"Voir le panier","cart_url":"https://www.pharmaly-dz.com/panier/","is_cart":"","cart_redirect_after_add":"no"}
----------------------------------------

Script 25
Keywords found: ['admin-ajax.php']
var woocommerce_params={"ajax_url":"/wp-admin/admin-ajax.php","wc_ajax_url":"/?wc-ajax=%%endpoint%%","i18n_password_show":"Afficher le mot de passe","i18n_password_hide":"Masquer le mot de passe"}
----------------------------------------

Script 27
Keywords found: ['admin-ajax.php']
var jetMenuPublicSettings={"version":"3.0.2.1","ajaxUrl":"https://www.pharmaly-dz.com/wp-admin/admin-ajax.php","isMobile":"false","getElementorTemplateApiUrl":"https://www.pharmaly-dz.com/wp-json/jet-menu-api/v2/get-elementor-template-content","getBlocksTemplateApiUrl":"https://www.pharmaly-dz.com/wp-json/jet-menu-api/v2/get-blocks-template-content"

In [51]:
for script in external_scripts:
    src = script.get("src", "")

    if "jet-engine" in src.lower():
        print(src)

### Dynamic Product Loading (AJAX)

The Santé category uses **JetEngine AJAX loading** to retrieve additional products when the user scrolls down.

The mechanism was identified through Chrome DevTools (Network → Fetch/XHR).

**Request parameters:**
- `action`: `jet_engine_ajax`
- `handler`: `listing_load_more`
- `query[query_id]`: `6`
- `query[posts_per_page]`: `10`
- `widget_settings[lisitng_id]`: `362`
- `widget_settings[load_more_type]`: `scroll`
- `page_settings[page]`: `2` (for the inspected request)

**Response structure:**

The server returns a JSON response containing:
- `success`: request status.
- `data.html`: HTML containing the next batch of product cards.
- `data.scripts` and `data.styles`: associated frontend resources.
- `data.fragments`: updated listing counters.

**Important observation:**

The dynamically returned product cards use the same HTML structure and CSS selectors as the product cards in the initial HTML response.

Therefore, the same BeautifulSoup extraction logic can be reused to process both initial and dynamically loaded products.

**Request details:**
- Request URL: `https://www.pharmaly-dz.com/categorie/sante/?nocache=1`
- HTTP method: `POST`
- Content-Type: `application/x-www-form-urlencoded; charset=UTF-8`

The request sends form-encoded parameters to identify the listing,
the query and the requested page.

In [60]:
first_card = next(iter(unique_cards.values()))

print(first_card.get_text(" ", strip=True))

Laboratoires NHCO Nutrition Effluvium Croissance Vitalité Du Cheveu 168 Gel Promo -1600 DA DA 11.500,00 Le prix initial était : DA 11.500,00. DA 9.900,00 Le prix actuel est : DA 9.900,00. J'ACHÈTE


In [61]:
product_url = first_card.select_one(
    "a.jet-listing-dynamic-image__link"
)["href"]

print(product_url)

https://www.pharmaly-dz.com/produit/effluvium-croissance-vitalite-du-cheveu-168-gel/


In [63]:
test_response = requests.get(
    "https://www.pharmaly-dz.com/",
    timeout=20
)

print("Status:", test_response.status_code)

Status: 200


In [64]:
product_response = requests.get(product_url, timeout=20)

print("Status:", product_response.status_code)
print("URL:", product_response.url)
print("HTML length:", len(product_response.text))


Status: 200
URL: https://www.pharmaly-dz.com/produit/effluvium-croissance-vitalite-du-cheveu-168-gel/
HTML length: 424381


In [65]:
product_soup = BeautifulSoup(product_response.text, "lxml")

print("Page title:", product_soup.title.get_text(strip=True))
print("Number of paragraphs:", len(product_soup.find_all("p")))
print("Number of divs:", len(product_soup.find_all("div")))

Page title: Effluvium Croissance Vitalité Du Cheveu 168 Gel | Pharmaly
Number of paragraphs: 22
Number of divs: 468


In [66]:
paragraphs = product_soup.find_all("p")

for i, paragraph in enumerate(paragraphs):
    text = paragraph.get_text(" ", strip=True)
    if text:
        print(f"Paragraph {i}: {text}")
        print("-" * 60)
        

Paragraph 0: DA 11.500,00 Le prix initial était : DA 11.500,00. DA 9.900,00 Le prix actuel est : DA 9.900,00.
------------------------------------------------------------
Paragraph 1: Complément capillaire qui stimule la croissance, renforce la fibre et améliore la vitalité des cheveux pour une chevelure plus dense et résistante .
------------------------------------------------------------
Paragraph 2: DA 11.500,00 Le prix initial était : DA 11.500,00. DA 9.900,00 Le prix actuel est : DA 9.900,00.
------------------------------------------------------------
Paragraph 3: Le Effluvium Croissance Vitalité du Cheveu 168 Gélules est un complément alimentaire capillaire formulé pour soutenir la croissance, la densité et la vitalité des cheveux . Grâce à une synergie de nutriments essentiels, il agit directement au niveau du bulbe capillaire pour renforcer la fibre et améliorer la qualité globale de la chevelure. Idéal pour les cheveux fins, fragilisés, ternes ou sujets à la chute , il contr

In [67]:
for i in [1, 3]:
    paragraph = paragraphs[i]

    print(f"Paragraph {i}")
    print("HTML:")
    print(paragraph.parent.prettify()[:2000])
    print("=" * 80)

Paragraph 1
HTML:
<div class="elementor-widget-container">
 <p>
  Complément capillaire qui stimule la croissance, renforce la fibre et améliore la vitalité des cheveux pour une
  <strong>
   chevelure plus dense et résistante
  </strong>
  .
 </p>
</div>

Paragraph 3
HTML:
<div aria-labelledby="elementor-tab-title-1771" class="elementor-tab-content elementor-clearfix" data-tab="1" id="elementor-tab-content-1771" role="region">
 <p>
  Le
  <strong>
   Effluvium Croissance Vitalité du Cheveu 168 Gélules
  </strong>
  est un complément alimentaire capillaire formulé pour soutenir la
  <strong>
   croissance, la densité et la vitalité des cheveux
  </strong>
  . Grâce à une synergie de nutriments essentiels, il agit directement au niveau du bulbe capillaire pour renforcer la fibre et améliorer la qualité globale de la chevelure. Idéal pour les cheveux
  <strong>
   fins, fragilisés, ternes ou sujets à la chute
  </strong>
  , il contribue à stimuler la croissance et à optimiser l’ancrage 

In [68]:
# Short description
short_element = product_soup.select_one(
    ".elementor-widget-container > p"
)

short_description = (
    short_element.get_text(" ", strip=True)
    if short_element else None
)

# Full description
full_element = product_soup.select_one(
    ".elementor-tab-content[data-tab='1']"
)

full_description = (
    full_element.get_text(" ", strip=True)
    if full_element else None
)

print("Short description:", short_description)
print("\nFull description:", full_description)

Short description: DA 11.500,00 Le prix initial était : DA 11.500,00. DA 9.900,00 Le prix actuel est : DA 9.900,00.

Full description: Le Effluvium Croissance Vitalité du Cheveu 168 Gélules est un complément alimentaire capillaire formulé pour soutenir la croissance, la densité et la vitalité des cheveux . Grâce à une synergie de nutriments essentiels, il agit directement au niveau du bulbe capillaire pour renforcer la fibre et améliorer la qualité globale de la chevelure. Idéal pour les cheveux fins, fragilisés, ternes ou sujets à la chute , il contribue à stimuler la croissance et à optimiser l’ancrage du cheveu. Utilisé en cure, il aide à retrouver une chevelure plus forte, plus épaisse et visiblement plus saine.


In [69]:
short_paragraph = paragraphs[1]

parent = short_paragraph.parent

while parent:
    print(
        "Tag:", parent.name,
        "| Classes:", parent.get("class"),
        "| ID:", parent.get("id")
    )

    parent = parent.parent

    if parent and parent.name == "body":
        break

Tag: div | Classes: ['elementor-widget-container'] | ID: None
Tag: div | Classes: ['elementor-element', 'elementor-element-34ce36a', 'elementor-widget', 'elementor-widget-text-editor'] | ID: None
Tag: div | Classes: ['elementor-element', 'elementor-element-226a3527', 'e-con-full', 'e-flex', 'e-con', 'e-child'] | ID: None
Tag: div | Classes: ['e-con-inner'] | ID: None
Tag: div | Classes: ['elementor-element', 'elementor-element-7dc812b7', 'jedv-enabled--yes', 'e-flex', 'e-con-boxed', 'e-con', 'e-parent'] | ID: None
Tag: div | Classes: ['elementor', 'elementor-1050', 'elementor-location-single', 'post-10413', 'product', 'type-product', 'status-publish', 'has-post-thumbnail', 'product_cat-shampoing-anti-chute', 'product_cat-complements-alimentaires-sante', 'marque_pharma-laboratoires-nhco-nutrition', 'first', 'instock', 'sale', 'shipping-taxable', 'purchasable', 'product-type-simple', 'product'] | ID: None


In [70]:
short_element = product_soup.select_one(
    ".elementor-widget-text-editor p"
)

short_description = (
    short_element.get_text(" ", strip=True)
    if short_element else None
)

print("Short description:", short_description)

Short description: Complément capillaire qui stimule la croissance, renforce la fibre et améliore la vitalité des cheveux pour une chevelure plus dense et résistante .


In [71]:
second_product_url = (
    "https://www.pharmaly-dz.com/produit/forcapil-anti-chute-90-comprimes/"
)

second_response = requests.get(second_product_url, timeout=20)

print("Status:", second_response.status_code)
print("HTML length:", len(second_response.text))

Status: 403
HTML length: 2356


In [72]:
print(second_response.text[:2000])

<!DOCTYPE html>
<html style="height:100%">
<head>
<meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no" />
<title> 403 Forbidden
</title><style>@media (prefers-color-scheme:dark){body{background-color:#000!important}}</style></head>
<body style="color: #444; margin:0;font: normal 14px/20px Arial, Helvetica, sans-serif; height:100%; background-color: #fff;">
<div style="height:auto; min-height:100%; ">     <div style="text-align: center; width:800px; margin-left: -400px; position:absolute; top: 30%; left:50%;">
        <h1 style="margin:0; font-size:150px; line-height:150px; font-weight:bold;">403</h1>
<h2 style="margin-top:20px;font-size: 30px;">Forbidden
</h2>
<p>Access to this resource on the server is denied!</p>
</div></div><div style="color:#f0f0f0; font-size:12px;margin:auto;padding:0px 30px 0px 30px;position:relative;clear:both;height:100px;margin-top:-101px;background-color:#474747;border-top: 1px solid rgba(0,0,0,0.15);box-shadow: 0 1px 0 rgba(2

In [73]:
second_product_url = "https://www.pharmaly-dz.com/produit/forcapil-anti-chute-90-comprmes/"

second_response = requests.get(second_product_url, timeout=20)

print("Status:", second_response.status_code)
print("HTML length:", len(second_response.text))

Status: 200
HTML length: 446587


In [74]:
second_soup = BeautifulSoup(second_response.text, "lxml")

# Short description
short_element = second_soup.select_one(
    ".elementor-widget-text-editor p"
)

short_description = (
    short_element.get_text(" ", strip=True)
    if short_element else None
)

# Full description
full_element = second_soup.select_one(
    ".elementor-tab-content[data-tab='1']"
)

full_description = (
    full_element.get_text(" ", strip=True)
    if full_element else None
)

print("Short description:", short_description)
print("\nFull description:", full_description)

Short description: Complément anti-chute qui fortifie les cheveux, réduit la perte capillaire et améliore la résistance et la densité de la fibre.

Full description: Forcapil Anti-Chute est un complément alimentaire conçu pour réduire la chute de cheveux réactionnelle et renforcer la fibre capillaire de l’intérieur. Il agit sur la qualité du cheveu en apportant les nutriments essentiels nécessaires à la synthèse de la kératine et au bon fonctionnement du follicule pileux. Il est particulièrement utilisé en cas de stress, fatigue, changement de saison ou déséquilibre passager pouvant fragiliser la chevelure. Sa formule associe des acides aminés soufrés, des vitamines et des minéraux qui aident à fortifier les cheveux, stimuler leur croissance et améliorer leur résistance. Une cure régulière permet généralement de retrouver des cheveux plus denses, plus forts et moins sujets à la casse.


### Product Detail Page Analysis

Product descriptions are not available in the product cards. They must be extracted from individual product pages.

Two types of descriptions were identified:

**1. Short description**

- HTML structure: `div.elementor-widget-text-editor > p`
- CSS selector: `.elementor-widget-text-editor p`
- Contains a brief summary of the product and its main benefits.

**2. Full description**

- HTML structure: `div.elementor-tab-content`
- Attribute: `data-tab="1"`
- CSS selector: `.elementor-tab-content[data-tab='1']`
- Contains more detailed information about the product, its properties and intended use.


Product extraction requires two stages:
1. Scrape category pages to collect product URLs and information available in product cards.
2. Visit individual product pages to extract short and full descriptions.

In [75]:
# Product name
print("Title:", product_soup.title.get_text(strip=True))

# Product images
images = product_soup.find_all("img")

print("\nNumber of images:", len(images))

for img in images[:5]:
    print("Alt:", img.get("alt"))
    print("Src:", img.get("src"))
    print("-" * 40)

Title: Effluvium Croissance Vitalité Du Cheveu 168 Gel | Pharmaly

Number of images: 16
Alt: None
Src: https://www.facebook.com/tr?id=24817307714546241&ev=PageView&noscript=1
----------------------------------------
Alt: 
Src: https://www.pharmaly-dz.com/wp-content/uploads/2024/03/logo-pharmaly-3.png
----------------------------------------
Alt: Effluvium Croissance Vitalité du Cheveu 168 Gel
Src: https://www.pharmaly-dz.com/wp-content/uploads/2024/11/effluvium-croissance-vitalite-du-cheveu-168-gel.webp
----------------------------------------
Alt: Effluvium Croissance Vitalité du Cheveu 168 Gel
Src: https://www.pharmaly-dz.com/wp-content/uploads/2024/11/effluvium-croissance-vitalite-du-cheveu-168-gel.webp
----------------------------------------
Alt: Thermoxyl Surpoids Bruleur de Graisses 112 Gel
Src: https://www.pharmaly-dz.com/wp-content/uploads/2024/11/thermoxyl-surpoids-bruleur-de-graisses-112-gel.jpg
----------------------------------------


In [79]:
for element in product_soup.find_all(["a", "span", "div"]):
    text = element.get_text(" ", strip=True)

    if text == "Laboratoires NHCO Nutrition":
        print("Tag:", element.name)
        print("Classes:", element.get("class"))
        print("HTML:")
        print(element.prettify()[:1500])
        print("-" * 50)
        

Tag: div
Classes: ['elementor-element', 'elementor-element-66846445', 'elementor-widget__width-inherit', 'elementor-widget', 'elementor-widget-jet-listing-dynamic-terms']
HTML:
<div class="elementor-element elementor-element-66846445 elementor-widget__width-inherit elementor-widget elementor-widget-jet-listing-dynamic-terms" data-e-type="widget" data-element_type="widget" data-id="66846445" data-widget_type="jet-listing-dynamic-terms.default">
 <div class="elementor-widget-container">
  <div class="jet-listing jet-listing-dynamic-terms">
   <a class="jet-listing-dynamic-terms__link" href="https://www.pharmaly-dz.com/marque_pharma/laboratoires-nhco-nutrition/">
    Laboratoires NHCO Nutrition
   </a>
  </div>
 </div>
</div>

--------------------------------------------------
Tag: div
Classes: ['elementor-widget-container']
HTML:
<div class="elementor-widget-container">
 <div class="jet-listing jet-listing-dynamic-terms">
  <a class="jet-listing-dynamic-terms__link" href="https://www.pha

In [80]:
brand_element = product_soup.select_one(
    ".jet-listing-dynamic-terms__link"
)

brand = brand_element.get_text(strip=True) if brand_element else None

print("Brand:", brand)

Brand: Laboratoires NHCO Nutrition


In [81]:
price_elements = product_soup.select("p.price")

print("Number of price elements:", len(price_elements))

for i, price in enumerate(price_elements):
    print(f"\nPrice element {i}:")
    print(price.prettify()[:2000])

Number of price elements: 10

Price element 0:
<p class="price">
 <del aria-hidden="true">
  <span class="woocommerce-Price-amount amount">
   <bdi>
    <span class="woocommerce-Price-currencySymbol" translate="no">
     DA
    </span>
    11.500,00
   </bdi>
  </span>
 </del>
 <span class="screen-reader-text">
  Le prix initial était : DA 11.500,00.
 </span>
 <ins aria-hidden="true">
  <span class="woocommerce-Price-amount amount">
   <bdi>
    <span class="woocommerce-Price-currencySymbol" translate="no">
     DA
    </span>
    9.900,00
   </bdi>
  </span>
 </ins>
 <span class="screen-reader-text">
  Le prix actuel est : DA 9.900,00.
 </span>
</p>


Price element 1:
<p class="price">
 <del aria-hidden="true">
  <span class="woocommerce-Price-amount amount">
   <bdi>
    <span class="woocommerce-Price-currencySymbol" translate="no">
     DA
    </span>
    11.500,00
   </bdi>
  </span>
 </del>
 <span class="screen-reader-text">
  Le prix initial était : DA 11.500,00.
 </span>
 <ins a

In [82]:
main_price = price_elements[0]

print(main_price.parent.parent.prettify()[:3000])

<div class="elementor-element elementor-element-b7b39cd elementor-widget__width-initial elementor-product-price-block-yes elementor-widget elementor-widget-woocommerce-product-price" data-e-type="widget" data-element_type="widget" data-id="b7b39cd" data-widget_type="woocommerce-product-price.default">
 <div class="elementor-widget-container">
  <p class="price">
   <del aria-hidden="true">
    <span class="woocommerce-Price-amount amount">
     <bdi>
      <span class="woocommerce-Price-currencySymbol" translate="no">
       DA
      </span>
      11.500,00
     </bdi>
    </span>
   </del>
   <span class="screen-reader-text">
    Le prix initial était : DA 11.500,00.
   </span>
   <ins aria-hidden="true">
    <span class="woocommerce-Price-amount amount">
     <bdi>
      <span class="woocommerce-Price-currencySymbol" translate="no">
       DA
      </span>
      9.900,00
     </bdi>
    </span>
   </ins>
   <span class="screen-reader-text">
    Le prix actuel est : DA 9.900,00.
   </

In [83]:
main_price = product_soup.select_one(
    ".elementor-widget-woocommerce-product-price p.price"
)

original_element = main_price.select_one("del") if main_price else None
current_element = main_price.select_one("ins") if main_price else None

if original_element and current_element:
    original_price = original_element.get_text(" ", strip=True)
    current_price = current_element.get_text(" ", strip=True)

elif main_price:
    original_price = None
    current_price = main_price.get_text(" ", strip=True)

else:
    original_price = None
    current_price = None

print("Original price:", original_price)
print("Current price:", current_price)

Original price: DA 11.500,00
Current price: DA 9.900,00


In [84]:
second_price = second_soup.select_one(
    ".elementor-widget-woocommerce-product-price p.price"
)

original_element = second_price.select_one("del") if second_price else None
current_element = second_price.select_one("ins") if second_price else None

if original_element and current_element:
    original_price = original_element.get_text(" ", strip=True)
    current_price = current_element.get_text(" ", strip=True)
elif second_price:
    original_price = None
    current_price = second_price.get_text(" ", strip=True)
else:
    original_price = None
    current_price = None

print("Original price:", original_price)
print("Current price:", current_price)

Original price: DA 9.000,00
Current price: DA 8.200,00


In [85]:
for product_id, card in unique_cards.items():
    price = card.select_one("p.price")

    if price and not price.select_one("del"):
        name_element = card.select_one("p.elementor-heading-title a")
        product_name = name_element.get_text(strip=True) if name_element else None

        product_url_element = card.select_one("a.jet-listing-dynamic-image__link")
        product_url = product_url_element.get("href") if product_url_element else None

        print("Product:", product_name)
        print("URL:", product_url)
        break

Product: Biogaia Gouttesprotectis   10ml
URL: https://www.pharmaly-dz.com/produit/biogaia-gouttesprotectis-10ml/


In [87]:
biogaia_response = requests.get(biogaia_url, timeout=20)

print("Status:", biogaia_response.status_code)
print("HTML length:", len(biogaia_response.text))

Status: 200
HTML length: 414214


In [88]:
biogaia_soup = BeautifulSoup(biogaia_response.text, "lxml")

biogaia_price = biogaia_soup.select_one(
    ".elementor-widget-woocommerce-product-price p.price"
)

original_element = biogaia_price.select_one("del") if biogaia_price else None
current_element = biogaia_price.select_one("ins") if biogaia_price else None

if original_element and current_element:
    original_price = original_element.get_text(" ", strip=True)
    current_price = current_element.get_text(" ", strip=True)
elif biogaia_price:
    original_price = None
    current_price = biogaia_price.get_text(" ", strip=True)
else:
    original_price = None
    current_price = None

print("Original price:", original_price)
print("Current price:", current_price)

Original price: None
Current price: DA 5.750,00


### Product Detail Page Field Extraction

Individual product pages provide additional information that is not available in category cards.

**1. Product image**
- Selector: `.jet-woo-product-gallery__image img`
- Attribute: `src`
- The main gallery image is used rather than smaller thumbnails or recommended product images.

**2. Brand**
- Selector: `.jet-listing-dynamic-terms__link`
- The brand is displayed as a link to its dedicated brand page.

**3. Prices**
- Selector: `.elementor-widget-woocommerce-product-price p.price`
- Original price: extracted from `<del>` when available.
- Current price: extracted from `<ins>` for promotional products.
- For regular products, the current price is extracted directly from the price element and the original price is set to `None`.

**4. Descriptions**
- Short description: `.elementor-widget-text-editor p`
- Full description: `.elementor-tab-content[data-tab='1']`

The product detail pages can provide the main product image, brand, prices and descriptions. Specific selectors are necessary to avoid extracting information from recommended products displayed on the same page.